## Benutils vs. PyTorch

### Question 
The purpose of this investigation is to understand and compare specific differences between my scratch network training engine, benutils, and the common open-source ML framework, PyTorch, under controlled conditions.

### Methodology
This investigation will be centered around training 2 neural network models on [Boston Housing](https://www.kaggle.com/code/prasadperera/the-boston-housing-dataset), a legacy testing dataset.

[The imported  dataset](https://www.kaggle.com/datasets/vikrishnan/boston-house-prices/data), is readily available to our models within this folder. It serves as a benchmark for model performance comparisons. I chose this established testing because of a history of consistent rigourous testing, evaluations, and familiarity that will help isolate variables for comparisons. 

For running and displaying output from benutils, the vanilla JavaScript library, I will utilize a Python subprocess and node.js.

##### Architecture
This investigation will compare performances for all 3 of benutils' basic architecture types: flat, power of two, and hourglass. Hourglass architecture will be using it's default operator, "/", leading to the standard contract-to-center into expanion-to-end shape that is associated with the hourglass name. 

However, before getting into full tests, I will create an extremely simple 2->2->1 model and manually match the weights of both to test basic functionality and math implementation.

In [ ]:
%%script node

import {neuralnet, set_features, set_features_csv, normalize_features} from "../src/network.js";

const model = neuralnet({
    hidden_layer_info: {
        count:1, 
        init_size:2, 
        style:"flat"
    },
    activation_type:"relu",
    learn_rate:.01,
    output_layer_info:{
        init_size:1, 
        activation:"none"
    },
    epochs:1,
    tt_split:0,
    random_seed:33463,
    batch_size:1
});

const features_in = {
    "feat1":[3.0, 12.0, 25.0],
    "feat2":[5.0, 15.0, 35.0],
    "TARGET":[22.0, 40.0, 11.0]
}

model.set_features(features_in);

model.train();


/* "THIS WAS TEMPORARILY PLACED IN model.train() to log initial weight and bias values."
    "Thankfully, due to seeding, it stays consistent. Initial output is saved as the output of this cell"
console.log("--input_layer--","\nvalues:", model.layers["input_layer"]["output"]);
console.log();
console.log("--hidden_layer_1--", "weights:", model.layers["hidden_layer_1"]["main"]["weights"], "\nbiases:", model.layers["hidden_layer_1"]["main"]["biases"], "\nvalues:", model.layers["hidden_layer_1"]["output"]);
console.log();
console.log("--output_layer--", "\nweights:", model.layers["output_layer"]["main"]["weights"], "\nbiases:", model.layers["output_layer"]["main"]["biases"], "\nvalues:", model.layers["output_layer"]["output"]);
*/


now buidling model[SEED: 33463]
1 epochs
0 train/test split
relu activation
1 batch size
flat architecture, hg_oper?: /
0.01 learning rate
--input_layer-- 
values: []

--hidden_layer_1-- weights: [
  [ -0.5771150797856148, 0.6096600024628742 ],
  [ 1.292116896082979, -0.41084944789670486 ]
] 
biases: [ 0, 0 ] 
values: []

--output_layer-- 
weights: [ [ 0.8596638505466694, -1.4355667459228387 ] ] 
biases: [ 0 ] 
values: []


Matching that initialization in PyTorch:

In [75]:
import torch
import numpy as np
import pandas as pd

class Network(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.input_layer = torch.nn.Linear(in_features=input_dim, out_features=2)
        self.hidden_layer = torch.nn.Linear(2, 1)

    def forward(self, x):
        x = torch.relu(self.input_layer(x))
        x = self.hidden_layer(x)
        return x

#MANUALLY MATCHING PARAMETER VALUES TO BENUTILS RUN
simple_input = torch.tensor([[3.0, 5.0], [12.0, 15.0], [25.0, 35.0]])
simple_targets = torch.tensor([[22.0], [40.0], [11.0]])

# 2 feature input layer
model = Network(input_dim=2)

custom_weights = torch.tensor([[0, 0], [0, 0]])
custom_bias = torch.tensor([0, 0])

with torch.no_grad():
    model.input_layer.weight.copy_(custom_weights)
    model.input_layer.bias.fill_(0)
    model.hidden_layer.bias.fill_(0)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

optimizer.zero_grad()

predictions = model(simple_input)

loss = criterion(predictions, simple_targets)

loss.backward()

weights_dict = model.state_dict()

for key in weights_dict.keys():
    print(key, weights_dict[key])

output = model(simple_input)
output



input_layer.weight tensor([[0., 0.],
        [0., 0.]])
input_layer.bias tensor([0., 0.])
hidden_layer.weight tensor([[-0.1771,  0.0621]])
hidden_layer.bias tensor([0.])


tensor([[0.],
        [0.],
        [0.]], grad_fn=<AddmmBackward0>)